# Fine-tuning réel par blocs — OmniASR CTC-1B Baoulé

Ce notebook entraîne 250 pas par session sur deux T4. Chaque bloc reprend l'état complet précédent, valide tous les 50 pas, sauvegarde un checkpoint complet à la fin et transcrit un panneau fixe du split `dev`. Le split `test` n'est jamais utilisé pour décider de poursuivre ou d'arrêter.

## 1. Aligner PyTorch sur fairseq2

In [ ]:
import subprocess
import sys
import torch

torch_base = torch.__version__.split('+')[0]
cuda_variant = f"cu{(torch.version.cuda or '').replace('.', '')}"
print('Python :', sys.version.split()[0], '| torch :', torch.__version__, '| variante :', cuda_variant)
if (torch_base, cuda_variant) != ('2.8.0', 'cu128'):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        'torch==2.8.0', 'torchaudio==2.8.0', 'torchvision==0.23.0',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
    raise RuntimeError('PyTorch 2.8.0 installé. Redémarre la session puis réexécute toutes les cellules depuis le début.')
print('PyTorch est compatible.')

## 2. Installer les dépôts et dépendances figés

In [ ]:
from pathlib import Path
import subprocess
import sys

META_COMMIT = '81f51e224ce9e74b02cc2a3eaf21b2d91d743455'
META_REPO = Path('/kaggle/working/omnilingual-asr')
PROJECT_REPO = Path('/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule')
PROJECT_URL = 'https://github.com/ngangoula-jaures/Finetune_omniASR_CTC_1B_for_baoule.git'
if not META_REPO.is_dir():
    subprocess.check_call(['git', 'clone', 'https://github.com/facebookresearch/omnilingual-asr.git', str(META_REPO)])
subprocess.check_call(['git', '-C', str(META_REPO), 'fetch', '--depth', '1', 'origin', META_COMMIT])
subprocess.check_call(['git', '-C', str(META_REPO), 'checkout', '--detach', META_COMMIT])
if not PROJECT_REPO.is_dir():
    subprocess.check_call(['git', 'clone', '--depth', '1', PROJECT_URL, str(PROJECT_REPO)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_REPO), 'pull', '--ff-only'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'fairseq2[arrow]==0.6.0',
    '--extra-index-url', 'https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    'datasets~=3.6.0', 'huggingface_hub>=0.32,<2', 'hf_xet', 'pyyaml',
    'pandas', 'soundfile>=0.13.1', 'pyarrow>=20,<22', 'polars>=1.29.0',
    'numba', 'librosa>=0.11.0', 'unidecode', 'uroman', 'fire', 'retrying',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', '-e', str(META_REPO)])
print('Installation terminée.')

In [ ]:
from importlib.metadata import version
import subprocess
import torch
import torchaudio

if torch.__version__.split('+')[0] != '2.8.0' or torchaudio.__version__.split('+')[0] != '2.8.0':
    raise RuntimeError(f'Environnement incompatible : torch={torch.__version__}, torchaudio={torchaudio.__version__}')
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(f'Il faut exactement deux GPU ; détectés : {torch.cuda.device_count()}')
for index in range(2):
    props = torch.cuda.get_device_properties(index)
    print(index, props.name, round(props.total_memory / 2**30, 2), 'Gio')
print('fairseq2 :', version('fairseq2'), '| omnilingual-asr :', version('omnilingual-asr'))
subprocess.run(['nvidia-smi'], check=True)

## 3. Télécharger la révision exacte du dataset

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient

HF_DATASET_ID = 'Tree-AI-lab/baoule-asr-dataset-mixture'
DATA_ROOT = Path('/kaggle/working/baoule_ctc1b_data')
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Le secret Kaggle HF_TOKEN est absent ou vide.')
os.environ['HF_HOME'] = '/kaggle/working/hf-cache'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
dataset_info = HfApi(token=HF_TOKEN).dataset_info(HF_DATASET_ID)
DATASET_REVISION = dataset_info.sha
if not DATASET_REVISION:
    raise RuntimeError('Révision Hugging Face introuvable.')
snapshot_download(
    repo_id=HF_DATASET_ID, repo_type='dataset', token=HF_TOKEN,
    revision=DATASET_REVISION, local_dir=DATA_ROOT,
)
DATASET_VERSION = DATA_ROOT / 'baoule_mixed' / 'version=0'
STATS_PATH = DATA_ROOT / 'language_distribution_0.tsv'
ASSET_CARD = DATA_ROOT / 'assets' / 'baoule_mixed.yaml'
for required in (DATASET_VERSION, STATS_PATH, ASSET_CARD):
    if not required.exists():
        raise FileNotFoundError(required)
print('Dataset :', f'{HF_DATASET_ID}@{DATASET_REVISION}')
del HF_TOKEN

## 4. Préflight du DataLoader

In [ ]:
from pathlib import Path
import subprocess
import sys

META_REPO = Path('/kaggle/working/omnilingual-asr')
DATA_ROOT = Path('/kaggle/working/baoule_ctc1b_data')
DATASET_VERSION = DATA_ROOT / 'baoule_mixed' / 'version=0'
STATS_PATH = DATA_ROOT / 'language_distribution_0.tsv'
subprocess.run([
    sys.executable, '-m', 'workflows.dataprep.dataloader_example',
    f'--dataset_path={DATASET_VERSION}', '--split=train', '--num_iterations=1',
    f'--stats_path={STATS_PATH}',
], cwd=META_REPO, check=True)

## 5. Définir le bloc

Pour le premier run, gardez `TARGET_STEP = 250`. Pour la session suivante, ajoutez la sortie Kaggle du bloc précédent comme **Input**, puis utilisez `TARGET_STEP = 500`.

In [ ]:
from pathlib import Path
import sys
import yaml

TARGET_STEP = 250
BLOCK_SIZE = 250
PREVIOUS_CHECKPOINT_DIR_OVERRIDE = None  # ex. Path('/kaggle/input/.../checkpoints')
if TARGET_STEP < BLOCK_SIZE or TARGET_STEP % BLOCK_SIZE != 0:
    raise ValueError('TARGET_STEP doit être un multiple positif de 250.')
START_STEP = TARGET_STEP - BLOCK_SIZE
PROJECT_REPO = Path('/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule')
sys.path.insert(0, str(PROJECT_REPO))
template_path = PROJECT_REPO / 'configs' / 'ctc_1b_stage.yaml'
runtime_config = Path('/kaggle/working') / f'ctc_1b_stage_to_{TARGET_STEP}.yaml'
config = yaml.safe_load(template_path.read_text(encoding='utf-8'))
config['regime']['num_steps'] = TARGET_STEP
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
assert config['lr_scheduler']['name'] == 'myle'
assert config['dataset']['asr_task_config']['max_audio_len'] == 320_000
assert config['dataset']['asr_task_config']['max_num_elements'] == 320_000
assert config['common']['no_sweep_dir'] is True
print(f'Bloc : {START_STEP} → {TARGET_STEP}')
print(runtime_config.read_text(encoding='utf-8'))

## 6. Préparer une sortie neuve ou la reprise du bloc précédent

In [ ]:
from pathlib import Path
import shutil
from src.stage_utils import find_checkpoint_dirs, prepare_resume_view

TRAIN_OUTPUT = Path('/kaggle/working/omniASR-CTC-1B-baoule-training')
if TRAIN_OUTPUT.exists() and any(TRAIN_OUTPUT.iterdir()):
    raise FileExistsError(f'Sortie non vide : {TRAIN_OUTPUT}. Utilise une session neuve.')
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)
RESUME_LINK = None
PREVIOUS_CHECKPOINT_DIR = None
if START_STEP > 0:
    if PREVIOUS_CHECKPOINT_DIR_OVERRIDE is not None:
        candidates = [Path(PREVIOUS_CHECKPOINT_DIR_OVERRIDE)]
    else:
        candidates = find_checkpoint_dirs(Path('/kaggle/input'), START_STEP)
    if len(candidates) != 1:
        listing = '\n'.join(map(str, candidates)) or '(aucun)'
        raise RuntimeError(f'Checkpoint précédent non déterminé :\n{listing}\nRenseigne PREVIOUS_CHECKPOINT_DIR_OVERRIDE.')
    PREVIOUS_CHECKPOINT_DIR = candidates[0]
    RESUME_LINK = prepare_resume_view(
        PREVIOUS_CHECKPOINT_DIR, TRAIN_OUTPUT / 'checkpoints', START_STEP,
    )
    previous_panel = PREVIOUS_CHECKPOINT_DIR.parent / 'dev_panel_manifest.json'
    if not previous_panel.is_file():
        raise FileNotFoundError(f'Panneau dev précédent absent : {previous_panel}')
    shutil.copy2(previous_panel, TRAIN_OUTPUT / previous_panel.name)
    print('Reprise en lecture seule :', RESUME_LINK, '→', RESUME_LINK.resolve())
else:
    print('Premier bloc : initialisation depuis omniASR_CTC_1B.')
free_gib = shutil.disk_usage('/kaggle/working').free / 2**30
print(f'Espace libre : {free_gib:.2f} Gio')
if free_gib < 16:
    raise RuntimeError('Moins de 16 Gio libres avant entraînement.')

## 7. Lancer le bloc distribué

In [ ]:
from pathlib import Path
import subprocess
import sys

runner = PROJECT_REPO / 'src' / 'run_training_stage.py'
stage_command = [
    sys.executable, str(runner), '--meta-repo', str(META_REPO),
    '--config', str(runtime_config), '--output-dir', str(TRAIN_OUTPUT),
    '--target-step', str(TARGET_STEP), '--expected-start-step', str(START_STEP),
    '--num-gpus', '2', '--dataset-id', HF_DATASET_ID,
    '--dataset-revision', DATASET_REVISION,
]
print('Commande :', ' '.join(stage_command))
subprocess.run(stage_command, check=True)

## 8. Vérifier le checkpoint puis détacher la reprise en lecture seule

In [ ]:
import json
from src.stage_utils import detach_resume_link

summary_path = TRAIN_OUTPUT / f'stage_{START_STEP:05d}_{TARGET_STEP:05d}_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
if summary['effective_return_code'] != 0 or not summary['target_checkpoint_complete']:
    raise RuntimeError('Bloc invalide : conserve les logs et ne détache pas le checkpoint précédent.')
detach_resume_link(RESUME_LINK)
print(f'Checkpoint complet du pas {TARGET_STEP} validé.')

## 9. Transcrire le panneau témoin `dev`

Le panneau contient deux exemples Waxal et deux exemples Klayt, choisis une seule fois. Son manifeste est conservé d'un bloc au suivant.

In [ ]:
from pathlib import Path
import subprocess
import sys

panel_manifest = TRAIN_OUTPUT / 'dev_panel_manifest.json'
panel_output = TRAIN_OUTPUT / f'dev_panel_predictions_step_{TARGET_STEP}.json'
transcribe_command = [
    sys.executable, str(PROJECT_REPO / 'src' / 'transcribe_dev_panel.py'),
    '--dataset-root', str(DATASET_VERSION),
    '--panel-manifest', str(panel_manifest), '--output', str(panel_output),
    '--checkpoint-dir', str(TRAIN_OUTPUT / 'checkpoints'),
    '--checkpoint-step', str(TARGET_STEP), '--examples-per-corpus', '2',
]
subprocess.run(transcribe_command, cwd=META_REPO, check=True)

In [ ]:
import json

panel = json.loads(panel_output.read_text(encoding='utf-8'))
print(f"Panneau — WER micro : {panel['micro_wer']:.2%} | CER micro : {panel['micro_cer']:.2%}")
for item in panel['examples']:
    print('\n', item['panel_id'], '|', item['corpus'], '|', f"{item['duration_seconds']:.1f}s")
    print('RÉF :', item['reference'])
    print('HYP :', item['prediction'])
    print(f"WER : {item['wer']:.2%} | CER : {item['cer']:.2%}")
print('\nUtilise maintenant Save Version avec les sorties activées.')

## 10. Décision

Après **Save Version**, transmettre le résumé du bloc, le log console et les prédictions du panneau. Ne pas lancer automatiquement le bloc suivant : la courbe WER, les overflows FP16 et les transcriptions doivent d'abord être examinés. Pour reprendre plus tard, ajouter cette sortie Kaggle comme Input et augmenter `TARGET_STEP` de 250.